In [0]:
# =============================================================================
# NOTEBOOK  : gold_fact_demand_trends
# PURPOSE   : gold.fact_daily_inventory → gold.fact_demand_trends
# LAYER     : Gold (demand analytics)
# FREQUENCY : Daily (after gold_fact_daily_inventory completes)
# WRITE     : Full overwrite partitioned by year_month
#             Rolling windows need full history — always full recompute
# GRAIN     : store_id + product_id + transaction_date
#
# LOGIC:
#   Reads fact_daily_inventory — which includes store shelf stock perspective
#   Computes rolling demand averages: 7d, 30d, 90d
#   Seasonality index = 7d avg / 90d avg (> 1.0 = selling faster than trend)
#   Trend direction   = 7d avg vs 30d avg (Rising / Falling / Stable)
#
# NOTE:
#   inventory_date = transaction_date cast to TimestampType
#   This matches PBI connected schema exactly (demand_intelligence table)
#   Rolling windows ordered by transaction_date (DateType)
#   revenue column from fact_daily_inventory = revenue (his column name)
# =============================================================================

# ── Imports & Config ──────────────────────────────────────────────────
import sys, json

_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"

sys.path.insert(0, PROJECT_ROOT)

from utils.audit import start_audit, end_audit
from utils.schema_registry import GOLD_FACT_DEMAND_TRENDS_SCHEMA

from pyspark.sql.functions import (
    current_timestamp, lit, col, round, when,
    avg, month, year, date_format
)
from pyspark.sql.window import Window

with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)

SOURCE_TABLE = cfg["tables"]["gold_fact_daily_inventory"]
TARGET_TABLE = cfg["tables"]["gold_fact_demand_trends"]
PIPELINE     = "gold_fact_demand_trends"

In [0]:
# ── Read + Compute + Write ───────────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, SOURCE_TABLE)

try:
    # Full read — rolling windows need complete history for accuracy
    fact = spark.read.table(SOURCE_TABLE)

    rows_read = fact.count()
    print(f"[READ] {rows_read} rows from {SOURCE_TABLE}")

    # Rolling windows partitioned by store+product
    # ordered by transaction_date (DateType) — our equivalent of his inventory_date
    w7  = (Window.partitionBy("store_id", "product_id")
                 .orderBy("transaction_date").rowsBetween(-7,  0))
    w30 = (Window.partitionBy("store_id", "product_id")
                 .orderBy("transaction_date").rowsBetween(-30, 0))
    w90 = (Window.partitionBy("store_id", "product_id")
                 .orderBy("transaction_date").rowsBetween(-90, 0))

    df = (
        fact

        # Rolling demand averages
        .withColumn("avg_units_7d",  round(avg("units_sold").over(w7),  2))
        .withColumn("avg_units_30d", round(avg("units_sold").over(w30), 2))
        .withColumn("avg_units_90d", round(avg("units_sold").over(w90), 2))

        # Rolling revenue averages
        # revenue in fact_daily_inventory = his revenue column, same name ✅
        .withColumn("avg_rev_7d",  round(avg("revenue").over(w7),  2))
        .withColumn("avg_rev_30d", round(avg("revenue").over(w30), 2))

        # Seasonality Index: 7-day avg / 90-day avg
        # > 1.0 = selling faster than 90-day trend (seasonal spike)
        .withColumn("seasonality_index",
            round(
                when(col("avg_units_90d") > 0,
                     col("avg_units_7d") / col("avg_units_90d"))
                .otherwise(lit(1.0)),
                3
            )
        )

        # Trend Direction: 7-day vs 30-day momentum
        # Rising  → selling 10%+ faster than last month
        # Falling → selling 10%+ slower than last month
        # Stable  → within ±10% of last month
        .withColumn("trend_direction",
            when(col("avg_units_7d") > col("avg_units_30d") * 1.1, "Rising")
            .when(col("avg_units_7d") < col("avg_units_30d") * 0.9, "Falling")
            .otherwise("Stable")
        )

        # Date parts for partitioning and seasonality analysis
        .withColumn("month_num",  month("transaction_date"))
        .withColumn("year_num",   year("transaction_date"))
        .withColumn("year_month", date_format("transaction_date", "yyyy-MM"))

        # inventory_date = transaction_date as timestamp
        # matches PBI connected schema exactly (demand_intelligence)
        .withColumn("inventory_date",
            col("transaction_date").cast("timestamp"))

        .withColumn("_gold_processed_at", current_timestamp())
        .withColumn("pipeline_run_id",    lit(run_id))

        .select([f.name for f in GOLD_FACT_DEMAND_TRENDS_SCHEMA.fields])
    )

    rows_written = df.count()

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("year_month")
        .option("overwriteSchema", "false")
        .saveAsTable(TARGET_TABLE)
    )

    print(f"[DONE] {rows_written} rows written to {TARGET_TABLE}")

    end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
              rows_read=rows_read, rows_written=rows_written)

except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── Verify ───────────────────────────────────────────────────────────
from pyspark.sql.functions import col

spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()).limit(1) \
    .select("status", "rows_read", "rows_written").show(truncate=False)

result = spark.read.table(TARGET_TABLE)
print("Row count:", result.count())
result.groupBy("trend_direction").count().show()
result.select(
    "store_id", "product_id", "inventory_date",
    "units_sold", "avg_units_7d", "avg_units_30d",
    "seasonality_index", "trend_direction"
).limit(5).show(truncate=False)